In [ ]:
import sys, os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'

import utils as ut
import meshio
import numpy as np
import pandas as pd
from pathlib import Path

In [ ]:
datadir = "./"
inname = "Nicoya_interface.e"
outname = "nicoya_rotated.geo"
geofile = datadir + outname

In [ ]:
mesh = meshio.read(datadir + inname)
points = mesh.points  # shape (n_points, 3)
print(mesh.cells)

In [ ]:
# ROTATION: Apply counterclockwise 45-degree rotation to points
print("Original points shape:", points.shape)
print("Original point range (before rotation):")
print(f"x: [{points[:,0].min():.3f}, {points[:,0].max():.3f}]")
print(f"y: [{points[:,1].min():.3f}, {points[:,1].max():.3f}]")
print(f"z: [{points[:,2].min():.3f}, {points[:,2].max():.3f}]")

# Define rotation matrix for 45 degrees counterclockwise
angle_deg = 45
angle_rad = np.radians(angle_deg)
cos_a = np.cos(angle_rad)
sin_a = np.sin(angle_rad)

rotation_matrix = np.array([
    [cos_a, -sin_a],
    [sin_a,  cos_a]
])

print(f"\nApplying {angle_deg}° counterclockwise rotation...")
print("Rotation matrix:")
print(rotation_matrix)

# Apply rotation to x,y coordinates (keeping z unchanged)
xy_original = points[:, :2].T  # Shape: (2, n_points)
xy_rotated = rotation_matrix @ xy_original
points_rotated = np.column_stack([xy_rotated.T, points[:, 2]])

print("\nRotated point range:")
print(f"x: [{points_rotated[:,0].min():.3f}, {points_rotated[:,0].max():.3f}]")
print(f"y: [{points_rotated[:,1].min():.3f}, {points_rotated[:,1].max():.3f}]")
print(f"z: [{points_rotated[:,2].min():.3f}, {points_rotated[:,2].max():.3f}]")

# Use rotated points for the rest of the process
points = points_rotated

In [ ]:
# Create DataFrame with rotated points
df = pd.DataFrame(points, columns=["x", "y", "z"])

# Print ranges
ranges = df.agg(["min", "max"])
print("DataFrame ranges (rotated coordinates):")
print(ranges)

In [ ]:
df['lon'], df['lat'] = ut.ckm2LLd(df['x'], df['y'], -84, 7, -45)

In [ ]:
# Import GNSS data
datadir_gnss = "/home/staff/chao/SSEinv/Nicoya/data/"
obs_disp_name = "Kyriakopoulos_novolcano.csv"

try:
    data = pd.read_csv(datadir_gnss + obs_disp_name, sep=",", skiprows=1, 
                       names=['site', 'lat', 'lon', 'height', 'Dt', 'Days', 
                              'vy_ITRF05', 'vy_std_ITRF05', 'vx_ITRF05', 'vx_std_ITRF05', 'vz_ITRF05', 'vz_std_ITRF05', 
                              'vy_Car', 'vy_std_Car', 'vx_Car', 'vx_std_Car', 'vz_Car', 'vz_std_Car'])
    data['lon'] = -1*data['lon'] # convert to east positive
    data['x'], data['y'] = ut.LL2ckmd(data['lon'], data['lat'], -84, 7, 45)
    
    # Apply the same rotation to GNSS data for consistency
    xy_gnss = data[['x', 'y']].values.T
    xy_gnss_rotated = rotation_matrix @ xy_gnss
    data['x'] = xy_gnss_rotated[0]
    data['y'] = xy_gnss_rotated[1]
    
except FileNotFoundError:
    print(f"GNSS data file not found: {datadir_gnss + obs_disp_name}")
    data = None

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.subplot(1, 2, 1)
plt.scatter(df['lon'], df['lat'], s=10, color='blue', marker='o', label='Fault points')
if data is not None:
    plt.scatter(data['lon'], data['lat'], s=10, color='red', marker='^', label='GNSS stations')
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Geographic coordinates")
plt.axis("equal")
plt.grid(True)
plt.legend()

plt.subplot(1, 2, 2)
plt.scatter(df['x']/1000, df['y']/1000, s=10, color='blue', marker='o', label='Fault points')
if data is not None:
    plt.scatter(data['x']/1000, data['y']/1000, s=10, color='red', marker='^', label='GNSS stations')
plt.xlabel("x (km, rotated 45° CCW)")
plt.ylabel("y (km, rotated 45° CCW)")
plt.title("Rotated Cartesian coordinates")
plt.axis("equal")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Define sep
sep = "//+"

# Define the cuboid boundary points, in meters
# Extend boundaries to accommodate rotated coordinates
xmin, xmax = -1500e3, 1500e3  # Extended to handle rotation
ymin, ymax = -1500e3, 1500e3  # Extended to handle rotation
zmin, zmax = -400e3, 0.0
print(f"Domain bounds: x=[{xmin}, {xmax}], y=[{ymin}, {ymax}], z=[{zmin}, {zmax}]")

# Calculate offset based on rotated coordinates
x0 = (df['x'].min() + df['x'].max()) / 2
y0 = (df['y'].min() + df['y'].max()) / 2
print(f"Calculated offset: x0={x0:.1f}, y0={y0:.1f}")

# write to the .geo file 
geo_content = f"""\
{sep}
xmin = {xmin};
xmax = {xmax};
ymin = {ymin};
ymax = {ymax};
zmin = {zmin};
zmax = {zmax}; 

"""

# Write to the file
with open(geofile, "w") as f:
    f.write(geo_content)

# Define the mesh sizes for different regions, in meters
dx = 200e3
dx_fault = 5e3
dx_fault_coarse = 20e3
dx_anomaly = 5e3
dx_anomaly_coarse = 20e3

geo_content = f"""\
{sep}
dx = {dx};
dx_fault = {dx_fault};
dx_fault_coarse = {dx_fault_coarse};
dx_anomaly = {dx_anomaly};
dx_anomaly_coarse = {dx_anomaly_coarse};

"""

# append to the file
with open(geofile, "a") as f:
    f.write(geo_content)

In [ ]:
# Same break points as original (these refer to point indices in the exodus file)
break1_list = [
    9, 10, 293, 437, 581, 725, 869,
    1013, 1157, 1301, 1445, 1589, 1733, 1877, 2021, 2165, 2309, 2453, 2597, 2741, 2885, 3029, 3173,
    3317, 3461, 3605, 3749, 3893, 4037, 4181, 4325, 4469, 4613, 4757, 4901, 5045, 5189, 5333, 5477,
    5621, 5765, 5909, 6053, 6197, 6341, 6485, 6629, 6773, 6917, 7061, 7205, 7349, 7493, 7637, 7781,
    7925, 8069, 8213, 8357, 8501, 8645, 8789, 8933, 9077, 9221, 9365, 9509
]
break2_list = [
    177, 178, 377, 521, 665, 809, 953,
    1097, 1241, 1385, 1529, 1673, 1817, 1961, 2105, 2249, 2393, 2537, 2681, 2825, 2969, 3113, 3257,
    3401, 3545, 3689, 3833, 3977, 4121, 4265, 4409, 4553, 4697, 4841, 4985, 5129, 5273, 5417, 5561,
    5705, 5849, 5993, 6137, 6281, 6425, 6569, 6713, 6857, 7001, 7145, 7289, 7433, 7577, 7721, 7865,
    8009, 8153, 8297, 8441, 8585, 8729, 8873, 9017, 9161, 9305, 9449, 9593
]
end3_list = [
    287, 288, 432, 576, 720, 864, 1008,
    1152, 1296, 1440, 1584, 1728, 1872, 2016, 2160, 2304, 2448, 2592, 2736, 2880, 3024, 3168, 3312,
    3456, 3600, 3744, 3888, 4032, 4176, 4320, 4464, 4608, 4752, 4896, 5040, 5184, 5328, 5472, 5616,
    5760, 5904, 6048, 6192, 6336, 6480, 6624, 6768, 6912, 7056, 7200, 7344, 7488, 7632, 7776, 7920,
    8064, 8208, 8352, 8496, 8640, 8784, 8928, 9072, 9216, 9360, 9504, 9648
]
start1_list = [1, 4] + [x + 2 for x in end3_list[1:-1]]

print(f"Break lists lengths: {len(break1_list)}, {len(break2_list)}, {len(end3_list)}, {len(start1_list)}")

In [ ]:
def exo_fault_to_points_with_dx_rotated(points, break1_list, break2_list,
                                        geo_file=None,
                                        x_offset=0, y_offset=0,
                                        coarse_str="dx_fault_coarse",
                                        fine_str="dx_fault"):
    """
    Modified version that uses pre-rotated points and handles boundary intersections better
    """
    if geo_file is None:
        raise ValueError("geo_file must be specified")

    seg2_points = set()
    for idx, (b1, b2) in enumerate(zip(break1_list, break2_list), start=1):
        if idx in (1, 2):
            seg2_points.update(range(b1, b2 + 1, 2))  # step=2 for lines 1 & 2
        else:
            seg2_points.update(range(b1, b2 + 1))      # normal contiguous

    with open(geo_file, "a") as f:
        for i, (x, y, z) in enumerate(points, start=1):
            dx_str = fine_str if i in seg2_points else coarse_str
            f.write(f"Point({i}) = {{{x-x_offset}, {y-y_offset}, {z}, {dx_str}}};\n")

    print(f"Wrote {geo_file} with {len(points)} points "
          f"({len(seg2_points)} points got '{fine_str}')")
    
    return len(points)

In [ ]:
npts = exo_fault_to_points_with_dx_rotated(
    points=points,
    break1_list=break1_list,
    break2_list=break2_list,
    geo_file=geofile,
    x_offset=x0,
    y_offset=y0,
    coarse_str="dx_fault_coarse",
    fine_str="dx_fault"
)

In [ ]:
# Create splines with the same logic as original
start_line_id = 1

# Loop through all entries
for i, (b1, b2, e3) in enumerate(zip(break1_list, break2_list, end3_list)):
    line_id = start_line_id + i
    base_id = (line_id - 1) * 3 + 1

    if i == 0:
        # Segment 1 expression
        seg1_expr = f"1, 2, 5:2:9"
        # Segment 2 expression
        seg2_expr = f"9:2:177"
        # Segment 3 expression
        seg3_expr = f"177:2:287"

    elif i == 1:    
        # Segment 1 expression
        seg1_expr = f"4, 3, 6:2:10"
        # Segment 2 expression
        seg2_expr = f"10:2:178"
        # Segment 3 expression
        seg3_expr = f"178:2:288"

    else:    
        # Segment 1 expression
        seg1_expr = f"{b1-3}, {b1-4}, {b1-2}:{b1}"
        # Segment 2 expression
        seg2_expr = f"{b1}:{b2}"
        # Segment 3 expression
        seg3_expr = f"{b2}:{e3}"

    ut.append_spline_to_geo(geofile, base_id, seg1_expr)
    ut.append_spline_to_geo(geofile, base_id + 1, seg2_expr)
    ut.append_spline_to_geo(geofile, base_id + 2, seg3_expr)

In [ ]:
# Update point and line count
start_point_id = npts + 1
start_line_id = (len(break1_list) * 3) + 1
print(f"Next line ID: {start_line_id}")

In [ ]:
# Create vertical lines connecting fault segments
vertical_line_start_ids = []
vertical_line_break1_ids = []
vertical_line_break2_ids = []
vertical_line_end3_ids = []

with open(geofile, "a") as f:
    # start1_list connections
    for i in range(len(start1_list) - 1):
        start_line_id += 1
        vertical_line_start_ids.append(start_line_id)
        f.write(f"Line({start_line_id}) = {{{start1_list[i]}, {start1_list[i+1]}}};\n")

    # break1_list connections
    for i in range(len(break1_list) - 1):
        start_line_id += 1
        vertical_line_break1_ids.append(start_line_id)
        f.write(f"Line({start_line_id}) = {{{break1_list[i]}, {break1_list[i+1]}}};\n")

    # break2_list connections
    for i in range(len(break2_list) - 1):
        start_line_id += 1
        vertical_line_break2_ids.append(start_line_id)
        f.write(f"Line({start_line_id}) = {{{break2_list[i]}, {break2_list[i+1]}}};\n")

    # end3_list connections
    for i in range(len(end3_list) - 1):
        start_line_id += 1
        vertical_line_end3_ids.append(start_line_id)
        f.write(f"Line({start_line_id}) = {{{end3_list[i]}, {end3_list[i+1]}}};\n")

In [ ]:
# Build curve loops and surfaces
surface_id = 1
curve_loop_id = 1

with open(geofile, "a") as f:
    for i in range(len(break1_list)-1):  # loop until second-to-last
        base_id = i * 3 + 1
        next_base_id = (i + 1) * 3 + 1

        # --- Segment 1 surface ---
        f.write(f"Curve Loop({curve_loop_id}) = "
                f"{{{base_id}, {vertical_line_break1_ids[i]}, -{next_base_id}, -{vertical_line_start_ids[i]}}};\n")
        f.write(f"Surface({surface_id}) = {{{curve_loop_id}}};\n")
        curve_loop_id += 1
        surface_id += 1

        # --- Segment 2 surface ---
        f.write(f"Curve Loop({curve_loop_id}) = "
                f"{{{base_id+1}, {vertical_line_break2_ids[i]}, -{next_base_id+1}, -{vertical_line_break1_ids[i]}}};\n")
        f.write(f"Surface({surface_id}) = {{{curve_loop_id}}};\n")
        curve_loop_id += 1
        surface_id += 1

        # --- Segment 3 surface ---
        f.write(f"Curve Loop({curve_loop_id}) = "
                f"{{{base_id+2}, {vertical_line_end3_ids[i]}, -{next_base_id+2}, -{vertical_line_break2_ids[i]}}};\n")
        f.write(f"Surface({surface_id}) = {{{curve_loop_id}}};\n")
        curve_loop_id += 1
        surface_id += 1

In [ ]:
# Add boundary completion - ymax boundary
def add_boundary_completion(points, geofile, xmin, xmax, ymin, ymax, zmin, zmax,
                           start1_list, end3_list, start_point_id, start_line_id,
                           curve_loop_id, surface_id, dx_str="dx"):
    """
    Add proper boundary completion to ensure the domain is fully enclosed
    """
    
    # Get the extreme points for boundary intersections
    start_x_coords = [points[pid-1][0] for pid in start1_list]
    end_x_coords = [points[pid-1][0] for pid in end3_list]
    
    # Find boundary intersection points
    start_xmin = min(start_x_coords)
    start_xmax = max(start_x_coords)
    end_xmin = min(end_x_coords)
    end_xmax = max(end_x_coords)
    
    boundary_points = []
    
    with open(geofile, "a") as f:
        # Add corner points for complete domain enclosure
        corners = [
            (xmin, ymax, zmax, "Top-left corner"),
            (xmax, ymax, zmax, "Top-right corner"),
            (xmax, ymin, zmax, "Bottom-right corner"),
            (xmin, ymin, zmax, "Bottom-left corner"),
            (xmin, ymax, zmin, "Top-left bottom"),
            (xmax, ymax, zmin, "Top-right bottom"),
            (xmax, ymin, zmin, "Bottom-right bottom"),
            (xmin, ymin, zmin, "Bottom-left bottom"),
        ]
        
        f.write(f"\n// Boundary corner points\n")
        for i, (x, y, z, desc) in enumerate(corners):
            pid = start_point_id + i
            f.write(f"Point({pid}) = {{{x}, {y}, {z}, {dx_str}}}; // {desc}\n")
            boundary_points.append(pid)
    
    print(f"Added {len(boundary_points)} boundary corner points")
    return boundary_points, start_point_id + len(boundary_points)

# Apply boundary completion
boundary_points, start_point_id = add_boundary_completion(
    points, geofile, xmin, xmax, ymin, ymax, zmin, zmax,
    start1_list, end3_list, start_point_id, start_line_id,
    curve_loop_id, surface_id
)

In [ ]:
# Read and add trench data (if available)
trench_file = datadir + "trench_xyz.txt"
try:
    trench = pd.read_csv(trench_file, 
                           delim_whitespace=True,
                           header=None,
                           names=["x", "y"])
    
    # Apply the same rotation to trench data
    xy_trench = trench[['x', 'y']].values.T
    xy_trench_rotated = rotation_matrix @ xy_trench
    trench['x'] = xy_trench_rotated[0]
    trench['y'] = xy_trench_rotated[1]
    
    # Filter trench data to domain bounds
    y_range = [points[:, 1].min(), points[:, 1].max()]
    trench = trench[(trench["y"] >= y_range[0]*0.9) & (trench["y"] <= y_range[1]*1.1)].reset_index(drop=True)
    
    if len(trench) > 0:
        # Sort and add boundary points
        trench = trench.sort_values(by='y', ascending=False).reset_index(drop=True)
        
        # Add top and bottom boundary points
        t_bound = pd.DataFrame([{'x': trench['x'].iloc[0], 'y': ymax}])
        b_bound = pd.DataFrame([{'x': trench['x'].iloc[-1], 'y': ymin}])
        trench = pd.concat([t_bound, trench, b_bound], ignore_index=True)
        
        # Write trench points
        with open(geofile, "a") as f:
            f.write(f"\n// Trench points (rotated)\n")
            for i, row in trench.iterrows():
                pid = start_point_id + i
                dx_size = "dx_fault_coarse" if i in [0, len(trench)-1] else "dx_fault"
                f.write(f"Point({pid}) = {{{row.x-x0:.6f}, {row.y-y0:.6f}, zmax, {dx_size}}}; // Trench point {i}\n")
        
        start_point_id += len(trench)
        print(f"Added {len(trench)} trench points (rotated)")
    else:
        print("No trench points in domain range after rotation")
        
except FileNotFoundError:
    print(f"Trench file not found: {trench_file}")
except Exception as e:
    print(f"Error processing trench data: {e}")

In [ ]:
print(f"\n=== Geo file creation completed ===\n")
print(f"Output file: {geofile}")
print(f"Total fault points: {npts}")
print(f"Applied 45° counterclockwise rotation to undo original strike/dip alignment")
print(f"Domain bounds extended and boundary intersections improved")
print(f"Next available point ID: {start_point_id}")
print(f"Next available line ID: {start_line_id}")